# New Approaches Tutorial

This tutorial demonstrates the new pharmacophore modeling approaches added to the toolkit:

1. **Ensemble Consensus** — Stability-aware clustering that runs multiple consensus rounds with perturbed parameters and votes on which features are robust
2. **Optimal Transport Scoring** — Wasserstein distance and Hungarian matching for pharmacophore comparison
3. **Strategy Selector** — Tournament-based selection across clustering and scoring strategies
4. **Multi-Fidelity Optimization** — Two-stage Bayesian optimization that explores cheaply then refines the best candidates

### Prerequisites

- Aligned reference molecules (we use CCR2 ligands from the main tutorial)
- Active and decoy molecule sets for evaluation
- `scikit-optimize` for Bayesian optimization features

## Setup & Data Loading

In [11]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, SDMolSupplier

from pharmacophore import Pharmacophore
from pharmacophore.consensus import PharmacophoreConsensus
from pharmacophore.mol_converter import PharmacophoreToMol
from pharmacophore.ensemble_consensus import EnsembleConsensus
from pharmacophore.hungarian_matching import match_features, pharmacophore_distance
from pharmacophore.ot_scoring import wasserstein_pharmacophore_distance, wasserstein_similarity
from pharmacophore.evaluation import UnifiedEvaluator, EvaluationConfig

In [12]:
# Load CCR2 reference ligands (pre-aligned from SDF)
refs = []
supplier = SDMolSupplier('data/CCR2_reference_ligands.sdf', removeHs=False)
for mol in supplier:
    if mol and mol.GetNumConformers() > 0:
        refs.append(mol)
print(f"Reference ligands: {len(refs)}")

# Load actives and decoys
actives_df = pd.read_csv('data/actives_ccr2_N75.csv')
decoys_df = pd.read_csv('data/decoys_ccr2_N500.csv')
active_smiles = actives_df['SMILES'].tolist()
decoy_smiles = decoys_df['Smiles'].tolist()
print(f"Actives: {len(active_smiles)}, Decoys: {len(decoy_smiles)}")

# Generate conformers for a small subset (for fast notebook execution)
def make_mol(smi, n_conf=3, seed=42):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    AllChem.EmbedMultipleConfs(mol, numConfs=n_conf, params=params)
    return mol if mol.GetNumConformers() > 0 else None

actives = [make_mol(s) for s in active_smiles]
actives = [m for m in actives if m is not None]

decoys = [make_mol(s) for s in decoy_smiles[:100]]  # subset for speed
decoys = [m for m in decoys if m is not None]

print(f"Prepared: {len(actives)} actives, {len(decoys)} decoys (3 conformers each)")

Reference ligands: 5
Actives: 74, Decoys: 498
Prepared: 74 actives, 100 decoys (3 conformers each)


## 1. Ensemble Consensus

Standard consensus pharmacophore generation uses a single set of parameters (tolerance, occurrence threshold). Small changes in these values can shift which features survive clustering.

**Ensemble Consensus** addresses this by running multiple consensus rounds with slightly perturbed parameters. Each feature gets a *stability score* — the fraction of rounds in which it appeared. Features that survive across many parameter variations are considered robust.

# New Approaches Tutorial

This tutorial demonstrates the new pharmacophore modeling approaches added to the toolkit:

1. **Ensemble Consensus** — Stability-aware clustering that runs multiple consensus rounds with perturbed parameters and votes on which features are robust
2. **Optimal Transport Scoring** — Wasserstein distance and Hungarian matching for pharmacophore comparison
3. **Strategy Selector** — Tournament-based selection across clustering and scoring strategies
4. **Multi-Fidelity Optimization** — Two-stage Bayesian optimization that explores cheaply then refines the best candidates

### Prerequisites

- Aligned reference molecules (we use CCR2 ligands from the main tutorial)
- Active and decoy molecule sets for evaluation
- `scikit-optimize` for Bayesian optimization features

In [13]:
# Create ensemble consensus with 25 perturbation rounds
ec = EnsembleConsensus(
    n_runs=25,
    tolerance_range=(1.5, 2.5),
    occurrence_range=(0.3, 0.7),
    stability_threshold=0.3,
    random_state=42,
)

features, stability_scores = ec.generate_consensus_with_scores(refs)

print(f"Ensemble consensus: {len(features)} features")
print(f"\n{'Type':<15} {'Stability':>10} {'Position'}")
print('-' * 55)
for feat, score in zip(features, stability_scores):
    print(f"{feat[0]:<15} {score:>10.2f}   ({feat[2]:.1f}, {feat[3]:.1f}, {feat[4]:.1f})")

Ensemble consensus: 8 features

Type             Stability Position
-------------------------------------------------------
Hydrophobe            0.68   (8.0, 32.3, 184.6)
Aromatic              0.64   (6.4, 30.4, 185.7)
Acceptor              0.52   (3.0, 29.1, 187.7)
Aromatic              0.52   (3.7, 26.2, 191.1)
Hydrophobe            0.44   (6.3, 25.8, 186.8)
Acceptor              0.40   (2.1, 27.7, 190.0)
Aromatic              0.36   (5.5, 27.5, 188.1)
Hydrophobe            0.36   (7.5, 24.9, 184.7)


In [14]:
# Compare with standard agglomerative consensus
standard = PharmacophoreConsensus(tolerance=2.0, occurrence_threshold=0.5)
standard_features = standard.generate_consensus(refs)

print(f"Standard consensus:  {len(standard_features)} features")
print(f"Ensemble consensus:  {len(features)} features")
print(f"  Highly stable (>0.7): {sum(1 for s in stability_scores if s >= 0.7)}")
print(f"  Moderately stable (0.4-0.7): {sum(1 for s in stability_scores if 0.4 <= s < 0.7)}")
print(f"  Low stability (<0.4): {sum(1 for s in stability_scores if s < 0.4)}")

Standard consensus:  7 features
Ensemble consensus:  8 features
  Highly stable (>0.7): 0
  Moderately stable (0.4-0.7): 6
  Low stability (<0.4): 2


## 2. Optimal Transport Scoring

Comparing two pharmacophore models requires matching their features. The toolkit provides two approaches:

- **Hungarian matching** — optimal 1-to-1 assignment that minimizes total distance (exact, fast)
- **Wasserstein distance** — earth mover's distance treating features as probability distributions (handles unequal feature counts naturally)

Both methods respect feature types: only features of the same type can be matched.

In [15]:
# Generate two pharmacophore models with different parameters
model_a = PharmacophoreConsensus(tolerance=1.5, occurrence_threshold=0.4)
features_a = model_a.generate_consensus(refs)

model_b = PharmacophoreConsensus(tolerance=2.5, occurrence_threshold=0.6)
features_b = model_b.generate_consensus(refs)

print(f"Model A (tol=1.5, occ=0.4): {len(features_a)} features")
print(f"Model B (tol=2.5, occ=0.6): {len(features_b)} features")

# Hungarian matching — optimal 1-to-1 assignment
pairs, unmatched_a, unmatched_b, total_cost = match_features(features_a, features_b)

# Compute per-pair spatial distances (cost also includes type penalty)
coords_a = np.array([[f[2], f[3], f[4]] for f in features_a], dtype=float)
coords_b = np.array([[f[2], f[3], f[4]] for f in features_b], dtype=float)
assignments = [
    (i, j, float(np.linalg.norm(coords_a[i] - coords_b[j])))
    for (i, j) in pairs
]

print(f"Hungarian matching: {len(assignments)} pairs, total cost = {total_cost:.2f}")
print(f"Unmatched A: {len(unmatched_a)}, Unmatched B: {len(unmatched_b)}")

for (i, j, dist) in assignments[:5]:
    print(f"  A[{i}] {features_a[i][0]:<12} <-> B[{j}] {features_b[j][0]:<12} dist={dist:.2f} A")



Model A (tol=1.5, occ=0.4): 13 features
Model B (tol=2.5, occ=0.6): 8 features
Hungarian matching: 8 pairs, total cost = 4.33
Unmatched A: 5, Unmatched B: 0
  A[0] Acceptor     <-> B[1] Acceptor     dist=0.30 A
  A[2] Acceptor     <-> B[2] Acceptor     dist=0.00 A
  A[3] Acceptor     <-> B[3] Acceptor     dist=0.48 A
  A[4] Acceptor     <-> B[0] Acceptor     dist=0.64 A
  A[5] Aromatic     <-> B[4] Aromatic     dist=0.00 A


In [16]:
# Wasserstein distance — earth mover's distance
w_dist = wasserstein_pharmacophore_distance(features_a, features_b)
w_sim = wasserstein_similarity(features_a, features_b)

# Hungarian distance for comparison
h_dist = pharmacophore_distance(features_a, features_b)

print(f"Wasserstein distance: {w_dist:.3f}")
print(f"Wasserstein similarity: {w_sim:.3f}")
print(f"Hungarian distance: {h_dist:.3f}")

# Self-distance should be 0
self_dist = wasserstein_pharmacophore_distance(features_a, features_a)
print(f"\nSelf-distance (sanity check): {self_dist:.6f}")

Wasserstein distance: 0.388
Wasserstein similarity: 0.612
Hungarian distance: 0.388

Self-distance (sanity check): 0.000000


## 3. Strategy Selector

Different datasets may benefit from different consensus and scoring strategies. The **StrategySelector** runs a tournament across multiple approaches and picks the best one based on screening performance.

Strategies include different combinations of:
- Clustering methods (agglomerative with various linkages, ensemble consensus)
- Parameter settings (tolerance, occurrence threshold)
- S_Dbw cluster validation scores

In [17]:
from pharmacophore.auto_strategy import StrategySelector

selector = StrategySelector(
    refs, actives, decoys,
    random_state=42,
    verbose=True,
)

best = selector.select_best()

print(f"\nBest strategy: {best.strategy_name}")
print(f"  ROC-AUC:    {best.eval_result.roc_auc:.4f}")
print(f"  BEDROC:     {best.eval_result.bedroc:.4f}")
print(f"  Features:   {best.eval_result.n_features}")
print(f"  S_Dbw:      {best.sdbw:.4f}" if best.sdbw != float('inf') else "  S_Dbw:      N/A")

  agglomerative        | tol=1.5 occ=0.3 | AUC=0.8131 BEDROC=0.9635 n_feat=13
  agglomerative        | tol=1.5 occ=0.5 | AUC=0.2596 BEDROC=0.0370 n_feat=7
  agglomerative        | tol=1.5 occ=0.7 | AUC=0.2053 BEDROC=0.0068 n_feat=5
  agglomerative        | tol=2.0 occ=0.3 | AUC=0.7893 BEDROC=0.9106 n_feat=12
  agglomerative        | tol=2.0 occ=0.5 | AUC=0.3928 BEDROC=0.0927 n_feat=7
  agglomerative        | tol=2.0 occ=0.7 | AUC=0.2912 BEDROC=0.0247 n_feat=4
  agglomerative        | tol=2.5 occ=0.3 | AUC=0.7895 BEDROC=0.9246 n_feat=11
  agglomerative        | tol=2.5 occ=0.5 | AUC=0.6130 BEDROC=0.5345 n_feat=8
  agglomerative        | tol=2.5 occ=0.7 | AUC=0.5516 BEDROC=0.2754 n_feat=5
  ensemble             | tol=1.5 occ=0.3 | AUC=0.6731 BEDROC=0.9318 n_feat=19
  ensemble             | tol=1.5 occ=0.5 | AUC=0.5645 BEDROC=0.2528 n_feat=11
  ensemble             | tol=1.5 occ=0.7 | AUC=0.4961 BEDROC=0.1581 n_feat=10
  ensemble             | tol=2.0 occ=0.3 | AUC=0.7612 BEDROC=0.9510 n_

## 4. Multi-Fidelity Optimization (Advanced)

The **ComboPharmacophoreOptimizer** supports multi-fidelity Bayesian optimization:

1. **Stage 1 (Exploration)**: Run BO with `n_conformers=1` per query molecule. This is fast but noisy — good for covering the search space.
2. **Stage 2 (Refinement)**: Take the top-K configurations from Stage 1 and re-evaluate them with full conformers.

This typically reduces wall time by 3-5x versus single-fidelity optimization while matching or exceeding final AUC.

In [18]:
from pharmacophore.combo_optimizer import ComboPharmacophoreOptimizer

optimizer = ComboPharmacophoreOptimizer(verbose=True, random_state=42)

# Load data using SMILES (the optimizer generates its own conformers)
ref_smiles = [Chem.MolToSmiles(m) for m in refs]
optimizer.load_from_smiles(
    reference_smiles=ref_smiles,
    active_smiles=active_smiles,
    decoy_smiles=decoy_smiles[:100],  # subset for speed
)

# Run multi-fidelity optimization with small budget
result = optimizer.optimize_multifidelity(
    n_calls=10,
    n_random_starts=5,
    n_conformers_final=5,
    explore_fraction=0.7,
    refine_top_k=3,
)

print(f"Best AUC: {result['best_auc']:.4f}")
print(f"Best params: {result['best_params']}")
print(f"Total evaluations: {result['n_evaluations']}")

# Stage timing breakdown
si = result.get('stage_info', {})
stage1_time = si.get('stage1_time_sec', si.get('explore_time_sec', 0.0))
stage2_time = si.get('stage2_time_sec', si.get('refine_time_sec', 0.0))
stage1_evals = si.get('stage1_evals', si.get('explore_evals', 0))
stage2_evals = si.get('stage2_evals', si.get('refine_configs', 0))

print(f"Stage 1 (explore):  {stage1_time:.1f}s, {stage1_evals} evals")
print(f"Stage 2 (refine):   {stage2_time:.1f}s, {stage2_evals} evals")



Loaded: 5 refs, 74 actives, 100 decoys
Multi-Fidelity Combo Pharmacophore Optimizer
References:      5
Actives:         74
Decoys:          100
Stage 1 budget:  7 evals (1 conformer)
Stage 2 refine:  top-3 (5 conformers)

--- Stage 1: Exploration ---
  [Explore   1/7] tmpl=3 tol=1.14 occ=0.80 opt=0.60 -> AUC=0.5000 (1-conf)
  [Explore   2/7] tmpl=2 tol=0.85 occ=0.51 opt=0.33 -> AUC=0.5000 (1-conf)


  [Explore   3/7] tmpl=0 tol=2.78 occ=0.15 opt=0.72 -> AUC=0.6689 (1-conf)
  [Explore   4/7] tmpl=4 tol=0.50 occ=0.99 opt=0.62 -> AUC=0.5000 (1-conf)


  [Explore   5/7] tmpl=3 tol=0.52 occ=0.12 opt=0.52 -> AUC=0.7932 (1-conf)


  [Explore   6/7] tmpl=4 tol=2.89 occ=0.10 opt=0.41 -> AUC=0.6835 (1-conf)


  [Explore   7/7] tmpl=4 tol=0.50 occ=0.10 opt=0.16 -> AUC=0.7695 (1-conf)

--- Stage 2: Refinement (top-3) ---


  [Refine 1/3] tmpl=3 tol=0.52 occ=0.12 opt=0.52 -> AUC=0.8532 (delta=+0.0600)


  [Refine 2/3] tmpl=4 tol=0.50 occ=0.10 opt=0.16 -> AUC=0.8268 (delta=+0.0573)


  [Refine 3/3] tmpl=4 tol=2.89 occ=0.10 opt=0.41 -> AUC=0.6900 (delta=+0.0065)



MULTI-FIDELITY OPTIMIZATION COMPLETE
Best AUC:        0.8532
Best params:     {'template_idx': 3, 'tolerance': 0.524732068269011, 'occurrence_threshold': 0.12075618253727419, 'opt_param': 0.5247746602583893, 'n_conformers': 5}
N features:      52
Explore evals:   7
Refine evals:    3
Total time:      52.9s
  Stage 1:       11.0s
  Stage 2:       41.9s
BEDROC(20):      0.5121460517495765
EF(1%):          0.0
Best AUC: 0.8532
Best params: {'template_idx': 3, 'tolerance': 0.524732068269011, 'occurrence_threshold': 0.12075618253727419, 'opt_param': 0.5247746602583893, 'n_conformers': 5}
Total evaluations: 10
Stage 1 (explore):  11.0s, 7 evals
Stage 2 (refine):   41.9s, 3 evals


In [19]:
# Compare with standard single-fidelity optimization (same budget)
result_sf = optimizer.optimize(
    n_calls=10,
    n_random_starts=5,
)

print(f"\nComparison (10 BO calls):")
print(f"  Multi-fidelity AUC: {result['best_auc']:.4f} ({result['elapsed_sec']:.1f}s)")
print(f"  Single-fidelity AUC: {result_sf['best_auc']:.4f} ({result_sf['elapsed_sec']:.1f}s)")

Combo Pharmacophore Optimizer
References:  5
Actives:     74
Decoys:      100
Evaluations: 10 (5 random starts)
Scoring:     rdShapeAlign combo Tanimoto (shape + color)
  [  1/10] tmpl=3 tol=1.14 occ=0.80 opt=0.60 nc=5 -> AUC=0.5000 (best=0.5000)


  [  2/10] tmpl=0 tol=2.11 occ=0.40 opt=0.14 nc=7 -> AUC=0.4412 (best=0.5000)
  [  3/10] tmpl=0 tol=3.03 occ=0.94 opt=0.00 nc=10 -> AUC=0.5000 (best=0.5000)


  [  4/10] tmpl=3 tol=2.64 occ=0.11 opt=0.02 nc=6 -> AUC=0.7658 (best=0.7658)
  [  5/10] tmpl=1 tol=0.66 occ=0.98 opt=0.23 nc=2 -> AUC=0.5000 (best=0.7658)
  [  6/10] tmpl=3 tol=3.91 occ=0.85 opt=0.00 nc=7 -> AUC=0.5000 (best=0.7658)


  [  7/10] tmpl=3 tol=0.79 occ=0.11 opt=0.03 nc=3 -> AUC=0.8454 (best=0.8454)
  [  8/10] tmpl=0 tol=3.21 occ=0.83 opt=0.05 nc=8 -> AUC=0.5000 (best=0.8454)


  [  9/10] tmpl=3 tol=0.50 occ=0.10 opt=0.05 nc=1 -> AUC=0.7730 (best=0.8454)


  [ 10/10] tmpl=3 tol=0.66 occ=0.30 opt=0.99 nc=5 -> AUC=0.1911 (best=0.8454)



OPTIMIZATION COMPLETE
Best AUC:      0.8454
Best params:   {'template_idx': 3, 'tolerance': 0.7949269855048031, 'occurrence_threshold': 0.10709411507424071, 'opt_param': 0.03090243719779322, 'n_conformers': 3}
N features:    50
Evaluations:   10
Elapsed:       51.2s
BEDROC(20):    0.5929689003795302
EF(1%):        0.0

Comparison (10 BO calls):
  Multi-fidelity AUC: 0.8532 (52.9s)
  Single-fidelity AUC: 0.8454 (51.2s)


## Summary

The table below compares the approaches demonstrated in this tutorial.

In [ ]:
# Build summary from the results above
evaluator = UnifiedEvaluator(refs, actives, decoys, random_state=42)

# Evaluate standard consensus
std_result = evaluator.evaluate_feature_subset(standard_features, 0.5, 0.5, 3)

# Evaluate ensemble consensus
ens_result = evaluator.evaluate_feature_subset(features, 0.5, 0.5, 3)

summary = pd.DataFrame([
    {'Approach': 'Standard Consensus', 'Features': len(standard_features),
     'ROC-AUC': f"{std_result.roc_auc:.4f}", 'BEDROC': f"{std_result.bedroc:.4f}"},
    {'Approach': 'Ensemble Consensus', 'Features': len(features),
     'ROC-AUC': f"{ens_result.roc_auc:.4f}", 'BEDROC': f"{ens_result.bedroc:.4f}"},
    {'Approach': f'Strategy Selector ({best.strategy_name})', 'Features': best.eval_result.n_features,
     'ROC-AUC': f"{best.eval_result.roc_auc:.4f}", 'BEDROC': f"{best.eval_result.bedroc:.4f}"},
    {'Approach': 'Multi-Fidelity BO', 'Features': len(result.get('best_features', [])),
     'ROC-AUC': f"{result['best_auc']:.4f}",
     'BEDROC': f"{result.get('best_metrics', {}).get('bedroc', 0.0):.4f}"},
])

summary